# Penalty calibration

The penalty is the single hyperparameter that controls the detection threshold of nearly every detector in Skchange: Larger penalty, fewer detections. Setting it well is what separates a detector that finds the interesting structure from one that either misses everything or floods the output with false alarms.

This page focuses on Monte-Carlo penalty calibration that controls the family-wise error rate (FWER), which is the probability of at least one false detection on a change-free series. The idea is to draw many change-free  or "null" samples, find the smallest `penalty_scale` that suppresses all detections on each, and take the `1 - level` quantile of those critical scales.

Skchange exposes this procedure through two entry points:

- [CalibratedDetector](../../api_reference/auto_generated/skchange.new_api.tuning.CalibratedDetector.rst): A scikit-learn-style meta-estimator that wraps a detector, calibrates its `penalty_scale` on `fit`, and delegates prediction to the calibrated detector. This is what you reach for in a normal workflow.
- [calibrate_penalty_scale](../../api_reference/auto_generated/skchange.new_api.tuning.calibrate_penalty_scale.rst): The underlying function returning the calibrated scale as a float. Use it when you want to compute the scale once and reuse it across many fits, or when you need to plug it into a custom pipeline.

The examples below use [SeededBinarySegmentation](../detectors/seeded_binseg.ipynb) with the [CUSUM](../../api_reference/auto_generated/skchange.new_api.interval_scorers.CUSUM.rst) statistic, but both tools work with any detector that exposes a scalar `penalty_scale`.

## Example data

We use the HVAC vibration dataset that ships with Skchange. It contains 30 days of 10-minute sampled vibration measurements from two HVAC units. Here we take the second unit's series as the target `X` to be analysed for regime changes.

In [ ]:
import warnings

import numpy as np
import plotly.express as px
import plotly.io as pio

from skchange.new_api.datasets import load_hvac_system_data

warnings.filterwarnings("ignore")
pio.renderers.default = "notebook"

hvac = load_hvac_system_data()
unit_ids = hvac["unit_id"]
selected_unit_id= np.unique(unit_ids)[1]
X = hvac["data"][unit_ids == selected_unit_id].reshape(-1, 1)
n_samples, n_features = X.shape
print(f"X shape: {X.shape}")

px.line(X.ravel(), labels={"index": "sample", "value": "vibration"}).show()

## Uncalibrated baseline

Every detector ships with a default penalty derived from a BIC-style formula in `n_samples` and `n_features`.
On series that have been standardised to approximately unit variance per segment,
and where the autocorrelation is not too strong, this default is often a good starting point.
Such standardisation, however, is not trivial on real data, and thus penalty calibration is almost always necessary.
Below we run the detector at the default (`penalty_scale=1.0`) as a reference point.
Due to the low scale of the series, with values between 0 and 0.08, the default penalty
is too strict, resulting in zero detections.

In [ ]:
from skchange.new_api.detectors import SeededBinarySegmentation
from skchange.new_api.interval_scorers import CUSUM
from skchange.new_api.utils.plotting import plot_detections

detector = SeededBinarySegmentation(CUSUM())
uncalibrated_cps = detector.fit_predict(X)
print(f"{len(uncalibrated_cps)} changepoints at penalty_scale=1.0")

plot_detections(X, changepoints=uncalibrated_cps).show()

## Calibrating with `CalibratedDetector`

`CalibratedDetector` wraps any detector that exposes a scalar `penalty_scale` and calibrates it for FWER control on `fit`. The main knobs are:

- `sampler`: How to draw change-free samples. Defaults to `"permutation"` — a non-parametric row-permutation of the fitted data that preserves the marginal distribution of each feature.
- `level`: Target FWER, i.e. the probability of at least one false detection. `0.05` is the standard choice.
- `n_simulations`: Number of null samples drawn. Larger gives a tighter estimate of the critical scale; the cost is linear.
- `random_state`: Seed for reproducibility. The RNG stream is invariant to `n_jobs`.
- `n_jobs`: Number of parallel workers for the Monte-Carlo loop. `-1` uses all cores.

Once fitted, the meta-estimator delegates `predict`, `predict_scores`, and friends to the calibrated inner detector, so you can drop it into any place that already accepts a detector.

In [ ]:
from skchange.new_api.tuning import CalibratedDetector

calibrated = CalibratedDetector(
    SeededBinarySegmentation(CUSUM()),
    sampler="permutation",
    level=0.05,
    n_simulations=200,
    random_state=0,
    n_jobs=-1,
).fit(X)

print(f"Calibrated penalty_scale: {calibrated.penalty_scale_:.3f}")

calibrated_cps = calibrated.predict(X)
print(f"{len(calibrated_cps)} changepoints at calibrated scale")

plot_detections(X, changepoints=calibrated_cps).show()

### Separate calibration data

By default the analysis data itself is used as the null source (each simulation permutes or resamples it). When the analysis data actually contains changes, this is conservative: the resampled variance overestimates the null variance and pushes the calibrated scale up. If you have a stretch of data you believe to be change-free, pass it as `X_calib` to `fit` to get a tighter calibration.

In [ ]:
from skchange.new_api.tuning import PermutationSampler

# A subrange of the same unit that we treat as change-free for illustration.
# The selection corresponds to the first five periods where the vibration is above 0.01.
X_calib = np.concatenate([X[296:365], X[443:504], X[584:650], X[730:790], X[873:937]])

calibrated_with_calib = CalibratedDetector(
    SeededBinarySegmentation(CUSUM()),
    sampler=PermutationSampler(replace=True),
    level=0.05,
    n_simulations=200,
    random_state=0,
    n_jobs=-1,
).fit(X, X_calib=X_calib)

print(
    f"penalty_scale_ (analysis-as-null): {calibrated.penalty_scale_:.3f}\n"
    f"penalty_scale_ (dedicated X_calib): {calibrated_with_calib.penalty_scale_:.3f}"
)

### Choosing a null sampler

The `sampler` argument controls how change-free samples are drawn. Skchange ships three named samplers:

- `"permutation"` (default, non-parametric): Resamples whole rows of the reference data without replacement. Preserves the joint marginal distribution across features while destroying temporal order. Assumes the underlying data is i.i.d. in time.
- `"gaussian"` (parametric): Draws i.i.d. N(0, 1). Ignores the reference data values. Gives the tightest calibration when the change-free distribution really is standard normal, and can be used with `X=None` to calibrate purely from shape.
- `"block_bootstrap"` (non-parametric): Copies contiguous blocks from the reference data, preserving short-range temporal dependence. Reach for it when the data is autocorrelated — an i.i.d. permutation would then misstate the false-alarm rate.

You can also pass a class with a `.sample` method, or just a function, with the signature
`(X, n_samples, random_state)` when you want to configure the sampler explicitly.
Built-in sampler classes are available in `skchange.new_api.tuning`.

In [ ]:
from skchange.new_api.tuning import BlockBootstrapSampler

calibrated_block = CalibratedDetector(
    SeededBinarySegmentation(CUSUM()),
    sampler=BlockBootstrapSampler(block_length=24),
    level=0.05,
    n_simulations=200,
    random_state=0,
    n_jobs=-1,
).fit(X, X_calib=X_calib)

print(
    f"penalty_scale_ (permutation):    {calibrated_with_calib.penalty_scale_:.3f}\n"
    f"penalty_scale_ (block bootstrap): {calibrated_block.penalty_scale_:.3f}"
)

## Calibrating once, reusing many times

`CalibratedDetector` recomputes the critical scale every time you call `fit`. If you want to calibrate once and then reuse the same scale across many datasets of a fixed shape (e.g. in a rolling deployment), call [calibrate_penalty_scale](../../api_reference/auto_generated/skchange.new_api.tuning.calibrate_penalty_scale.rst) directly and set `penalty_scale` on the detector yourself.

In [ ]:
from skchange.new_api.tuning import calibrate_penalty_scale

penalty_scale = calibrate_penalty_scale(
    SeededBinarySegmentation(CUSUM()),
    n_samples=n_samples,
    n_features=n_features,
    X=X_calib,
    sampler=BlockBootstrapSampler(block_length=24),
    level=0.05,
    n_simulations=200,
    random_state=0,
    n_jobs=-1,
)
print(f"Calibrated penalty_scale: {penalty_scale:.3f}")

# Reuse for many future series of the same shape.
reused_detector = SeededBinarySegmentation(CUSUM(), penalty_scale=penalty_scale)
reused_cps = reused_detector.fit_predict(X)
print(f"{len(reused_cps)} changepoints on the analysis series")

## Practical guidance

- `level`: The standard target is `0.05`. Use `0.01` when a false detection is expensive, `0.10` when missing a real change is worse.
- `n_simulations`: For a `1 - level` quantile you need at least a handful of null samples in the tail. As a rule of thumb, `n_simulations >= 20 / level`, so `400` for `level=0.05` and `2000` for `level=0.01`. The default `1000` is a good balance.
- **Compute cost**: Each null sample runs the detector's calibration procedure once.
  For `SeededBinarySegmentation`, `MovingWindow`, `CircularBinarySegmentation` that is a single `fit(X).predict_scores(X)` call; for `PELT` it is a few `fit_predict` calls (path search); and for `CAPA` it is roughly 15-25 `fit_predict` calls by the generic fallback method of finding a lowest penalty scale yielding zero detections. Parallelise with `n_jobs=-1`.
- **Autocorrelated data**: Use `sampler="block_bootstrap"` with a `block_length` on the scale of the strongest short-range dependence.
- **CROPS**: Cannot be calibrated with these tools because it does not have a single `penalty_scale` knob. Calibrate the underlying `PELT` instead, or use CROPS's own BIC / elbow selection.